# Sales Analysis — Jupyter Notebook
**Dataset:** Sales_transactions_2022_2025.csv  
**Period:** 2022 – 2025  
**Rows:** 18,045 transactions

---
## Steps
1. Collect and load the CSV dataset
2. Check for missing or incorrect values
3. Calculate Total Sales = Quantity × Unit Price
4. Group & Summarise (totals, counts, averages)
5. Create charts to compare results
6. Use results to make business decisions

## 0. Install & Import Libraries

In [ ]:
# Install required packages (run once if needed)
# !pip install pandas matplotlib seaborn

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# Plotting defaults
PALETTE = ['#3b82d4', '#7c5cd8', '#e85d3a', '#22c55e', '#f59e0b', '#06b6d4']
sns.set_theme(style='whitegrid', palette=PALETTE)
plt.rcParams.update({'font.family': 'DejaVu Sans',
                     'figure.figsize': (10, 5),
                     'axes.spines.top': False,
                     'axes.spines.right': False})

print('Libraries loaded successfully.')

## Step 1 — Collect and Load the Dataset

In [ ]:
df_raw = pd.read_csv('Sales_transactions_2022_2025.csv')

print(f'Dataset shape : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(f'Years covered : {sorted(df_raw["Order_Year"].unique())}')
df_raw.head()

## Step 2 — Data Quality Check

In [ ]:
print('=== Data Types ===')
print(df_raw.dtypes)

print('\n=== Missing Values ===')
missing = df_raw.isnull().sum()
missing_pct = (df_raw.isnull().mean() * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

In [ ]:
print('Unique Order_Status values (raw):', df_raw['Order_Status'].unique())
print('Unique Product_Category values (raw):', df_raw['Product_Category'].unique())

In [ ]:
# ── Visualise missing values ──────────────────────────────────────────────────
missing_plot = df_raw.isnull().sum()
missing_plot = missing_plot[missing_plot > 0].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(missing_plot.index, missing_plot.values, color=PALETTE[0])
ax.bar_label(bars, fmt='%d', padding=4, fontsize=9)
ax.set_xlabel('Missing Count', fontsize=11)
ax.set_title('Missing Values per Column (Raw Dataset)', fontsize=13, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ── Data Cleaning ─────────────────────────────────────────────────────────────
df = df_raw.copy()

# Standardise text columns to Title Case
for col in ['Order_Status', 'Product_Category', 'Product_Subcategory',
            'Sales_Channel', 'Customer_Segment', 'Customer_Gender',
            'Country', 'Return_Flag', 'Payment_Method']:
    df[col] = df[col].str.strip().str.title()

# Fill missing text
df['Return_Reason']  = df['Return_Reason'].fillna('N/A')
df['Promotion_Code'] = df['Promotion_Code'].fillna('None')
df['Customer_Name']  = df['Customer_Name'].fillna('Unknown')
df['Product_Name']   = df['Product_Name'].fillna('Unknown')
df['Payment_Method'] = df['Payment_Method'].fillna('Unknown')
df['Customer_Gender']= df['Customer_Gender'].fillna('Unknown')

# Fill numeric with median
for col in ['Customer_Age', 'Customer_Rating', 'Inventory_Level', 'Delivery_Days']:
    df[col] = df[col].fillna(df[col].median())

# Parse date and derive new columns
df['Order_Date']       = pd.to_datetime(df['Order_Date'], dayfirst=True, errors='coerce')
df['Order_Month']      = df['Order_Date'].dt.month
df['Order_Month_Name'] = df['Order_Date'].dt.strftime('%b')
df['Order_Quarter']    = df['Order_Date'].dt.quarter.map({1:'Q1',2:'Q2',3:'Q3',4:'Q4'})

print('Cleaning complete. Missing values after cleaning:')
print(df.isnull().sum()[df.isnull().sum() > 0])

## Step 3 — Calculate Total Sales = Quantity × Unit Price

In [ ]:
df['Total_Sales'] = df['Quantity'] * df['Unit_Price']

print('Total_Sales column added.')
print(f'Min  : ${df["Total_Sales"].min():,.2f}')
print(f'Max  : ${df["Total_Sales"].max():,.2f}')
print(f'Mean : ${df["Total_Sales"].mean():,.2f}')
print(f'Total: ${df["Total_Sales"].sum():,.2f}')
df[['Transaction_ID', 'Product_Name', 'Quantity', 'Unit_Price', 'Total_Sales']].head(10)

## Step 4 — Group & Summarise (Totals, Counts, Averages)

In [ ]:
# Overall summary
print('=== Overall KPIs ===')
print(f'Total Revenue   : ${df["Total_Sales"].sum():,.2f}')
print(f'Total Orders    : {df["Transaction_ID"].nunique():,}')
print(f'Avg Order Value : ${df["Total_Sales"].mean():,.2f}')
print(f'Total Units Sold: {df["Quantity"].sum():,}')
print(f'Return Rate     : {(df["Return_Flag"]=="Yes").mean()*100:.2f}%')
print(f'Avg Rating      : {df["Customer_Rating"].mean():.2f}/5')

In [ ]:
# By Year
year_summary = df.groupby('Order_Year').agg(
    Total_Revenue =('Total_Sales',    'sum'),
    Orders        =('Transaction_ID', 'count'),
    Avg_Order_Value=('Total_Sales',   'mean'),
    Total_Units   =('Quantity',        'sum'),
    Avg_Rating    =('Customer_Rating', 'mean'),
).reset_index()
year_summary

In [ ]:
# By Product Category
cat_summary = df.groupby('Product_Category').agg(
    Total_Revenue =('Total_Sales',    'sum'),
    Orders        =('Transaction_ID', 'count'),
    Avg_Sale      =('Total_Sales',    'mean'),
    Total_Qty     =('Quantity',        'sum'),
).sort_values('Total_Revenue', ascending=False).reset_index()
cat_summary

In [ ]:
# By Country
country_summary = df.groupby('Country').agg(
    Total_Revenue  =('Total_Sales',    'sum'),
    Orders         =('Transaction_ID', 'count'),
    Avg_Sale       =('Total_Sales',    'mean'),
).sort_values('Total_Revenue', ascending=False).reset_index()
country_summary

In [ ]:
# By Sales Channel
channel_summary = df.groupby('Sales_Channel').agg(
    Total_Revenue =('Total_Sales',    'sum'),
    Orders        =('Transaction_ID', 'count'),
    Avg_Sale      =('Total_Sales',    'mean'),
).sort_values('Total_Revenue', ascending=False).reset_index()
channel_summary

In [ ]:
# By Customer Segment
seg_summary = df.groupby('Customer_Segment').agg(
    Total_Revenue =('Total_Sales',    'sum'),
    Orders        =('Transaction_ID', 'count'),
    Avg_Sale      =('Total_Sales',    'mean'),
    Avg_Rating    =('Customer_Rating','mean'),
).sort_values('Total_Revenue', ascending=False).reset_index()
seg_summary

## Step 5 — Charts

In [ ]:
# Chart 1 — Annual Revenue Bar Chart
yr = df.groupby('Order_Year')['Total_Sales'].sum().reset_index()
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(yr['Order_Year'].astype(str), yr['Total_Sales']/1e6,
              color=PALETTE[:len(yr)], edgecolor='white')
ax.bar_label(bars, fmt='$%.2fM', padding=4, fontsize=10)
ax.set_xlabel('Year'); ax.set_ylabel('Revenue ($ Millions)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.1f}M'))
ax.set_title('Annual Revenue (2022–2025)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Chart 2 — Revenue by Product Category
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(cat_summary['Product_Category'], cat_summary['Total_Revenue']/1e6,
               color=PALETTE[:len(cat_summary)])
ax.bar_label(bars, fmt='$%.2fM', padding=4, fontsize=9)
ax.set_xlabel('Revenue ($ Millions)'); ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.1f}M'))
ax.set_title('Revenue by Product Category', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Chart 3 — Revenue by Country
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(country_summary['Country'], country_summary['Total_Revenue']/1e6,
              color=PALETTE[:len(country_summary)], edgecolor='white')
ax.bar_label(bars, fmt='$%.2fM', padding=4, fontsize=9)
ax.set_ylabel('Revenue ($ Millions)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.1f}M'))
ax.set_title('Revenue by Country', fontsize=14, fontweight='bold')
plt.xticks(rotation=20, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
# Chart 4 — Sales Channel Pie
fig, ax = plt.subplots(figsize=(6, 5))
ax.pie(channel_summary['Total_Revenue'], labels=channel_summary['Sales_Channel'],
       autopct='%1.1f%%', colors=PALETTE[:len(channel_summary)],
       startangle=140, wedgeprops={'edgecolor':'white','linewidth':1.5})
ax.set_title('Revenue Share by Sales Channel', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Chart 5 — Monthly Revenue Trend per Year
monthly = df.groupby(['Order_Year','Order_Month'])['Total_Sales'].sum().reset_index()
monthly['Month_Label'] = pd.to_datetime(monthly['Order_Month'],format='%m').dt.strftime('%b')

fig, ax = plt.subplots(figsize=(12, 5))
for i, yr in enumerate(sorted(monthly['Order_Year'].unique())):
    sub = monthly[monthly['Order_Year']==yr].sort_values('Order_Month')
    ax.plot(sub['Month_Label'], sub['Total_Sales']/1e3,
            marker='o', linewidth=2, label=str(yr), color=PALETTE[i])
ax.set_ylabel('Revenue ($ Thousands)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.0f}K'))
ax.set_title('Monthly Revenue Trend by Year', fontsize=14, fontweight='bold')
ax.legend(title='Year'); plt.tight_layout(); plt.show()

In [ ]:
# Chart 6 — Top 15 Products by Revenue
top15 = df.groupby('Product_Name')['Total_Sales'].sum().nlargest(15).reset_index()
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(top15['Product_Name'], top15['Total_Sales']/1e3, color=PALETTE[0])
ax.bar_label(bars, fmt='$%.1fK', padding=4, fontsize=9)
ax.set_xlabel('Revenue ($ Thousands)'); ax.invert_yaxis()
ax.set_title('Top 15 Products by Revenue', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Chart 7 — Customer Segment Revenue
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(seg_summary['Customer_Segment'], seg_summary['Total_Revenue']/1e6,
              color=PALETTE[:len(seg_summary)], edgecolor='white')
ax.bar_label(bars, fmt='$%.2fM', padding=4, fontsize=9)
ax.set_ylabel('Revenue ($ Millions)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.1f}M'))
ax.set_title('Revenue by Customer Segment', fontsize=14, fontweight='bold')
plt.xticks(rotation=25, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
# Chart 8 — Order Status Distribution
status_cnt = df['Order_Status'].value_counts()
fig, ax = plt.subplots(figsize=(7, 5))
ax.pie(status_cnt, labels=status_cnt.index, autopct='%1.1f%%',
       colors=PALETTE[:len(status_cnt)], startangle=140,
       wedgeprops={'edgecolor':'white','linewidth':1.5})
ax.set_title('Order Status Distribution', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Chart 9 — Return Rate by Category
ret = df.groupby('Product_Category').apply(
    lambda x: (x['Return_Flag']=='Yes').mean()*100
).reset_index(name='Return_Rate_%').sort_values('Return_Rate_%', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(ret['Product_Category'], ret['Return_Rate_%'],
              color=PALETTE[2], edgecolor='white')
ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=9)
ax.set_ylabel('Return Rate (%)'); ax.set_xlabel('Product Category')
ax.set_title('Return Rate by Product Category', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Chart 10 — Category Revenue by Country (Stacked Bar)
pivot = df.pivot_table(values='Total_Sales', index='Country',
                       columns='Product_Category', aggfunc='sum', fill_value=0)
fig, ax = plt.subplots(figsize=(10, 5))
pivot.div(1e6).plot(kind='bar', ax=ax, color=PALETTE[:len(pivot.columns)],
                   edgecolor='white', linewidth=0.5)
ax.set_ylabel('Revenue ($ Millions)'); ax.set_xlabel('Country')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.1f}M'))
ax.set_title('Category Revenue by Country', fontsize=14, fontweight='bold')
ax.legend(title='Category', bbox_to_anchor=(1,1), fontsize=9)
plt.xticks(rotation=20, ha='right'); plt.tight_layout(); plt.show()

## Step 6 — Business Decisions

In [ ]:
top_country  = df.groupby('Country')['Total_Sales'].sum().idxmax()
top_category = df.groupby('Product_Category')['Total_Sales'].sum().idxmax()
top_channel  = df.groupby('Sales_Channel')['Total_Sales'].sum().idxmax()
top_segment  = df.groupby('Customer_Segment')['Total_Sales'].sum().idxmax()
high_return_cat = df.groupby('Product_Category').apply(
    lambda x: (x['Return_Flag']=='Yes').mean()).idxmax()
best_month   = df.groupby('Order_Month')['Total_Sales'].sum().idxmax()
best_month_name = pd.to_datetime(best_month, format='%m').strftime('%B')
promo_avg    = df[df['Promotion_Code']!='None']['Total_Sales'].mean()
no_promo_avg = df[df['Promotion_Code']=='None']['Total_Sales'].mean()
promo_lift   = (promo_avg - no_promo_avg) / no_promo_avg * 100

print(f'1. Top Revenue Country     : {top_country}')
print(f'2. Top Revenue Category    : {top_category}')
print(f'3. Top Sales Channel       : {top_channel}')
print(f'4. Top Customer Segment    : {top_segment}')
print(f'5. Peak Revenue Month      : {best_month_name}')
print(f'6. Highest Return Category : {high_return_cat}')
print(f'7. Avg Sale (With Promo)   : ${promo_avg:,.2f}')
print(f'   Avg Sale (No Promo)     : ${no_promo_avg:,.2f}')
print(f'   Promo Lift              : {promo_lift:+.1f}%')
print(f'8. Avg Customer Rating     : {df["Customer_Rating"].mean():.2f}/5')

In [ ]:
# Business Decision Summary
decisions = {
    'Invest in top market': f'Increase marketing budget for {top_country}.',
    'Category focus':       f'Prioritise stock & upsells for {top_category}.',
    'Channel investment':   f'UX & loyalty programmes for {top_channel} channel.',
    'Segment programme':    f'VIP programme for {top_segment} customers.',
    'Seasonal planning':    f'Inventory build and campaigns ahead of {best_month_name}.',
    'Reduce returns':       f'QC & product content improvements for {high_return_cat}.',
    'Promotions':           f'Promotions drive a {promo_lift:+.1f}% avg sale lift — expand targeted offers.',
    'Rating improvement':   'Follow up on sub-3-star orders to improve NPS and retention.',
}

print('=== Strategic Business Decisions ===')
for k, v in decisions.items():
    print(f'  • {k}: {v}')